# Valuation & Metrics (SPINRec-style Step 3)

在三个基准模型上计算论文方法指标：SIF 与阶段分解 DVF。

In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("nbformat") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])

%run ./functions.ipynb

from pathlib import Path

def _find_root() -> Path:
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "DATASET").exists() and (p / "recacc").exists():
            return p
    return c

REPO_ROOT = _find_root()
seed_everything(42)

cache = torch.load(REPO_ROOT / "recacc" / "log" / "notebook_cache" / "prepared_data.pt")
prepared = PreparedData(
    dense=cache["dense"],
    sparse=cache["sparse"],
    labels=cache["labels"],
    sample_ids=cache["sample_ids"],
    sparse_fields=cache["sparse_fields"],
    dense_fields=cache["dense_fields"],
)
NUM_BUCKETS = int(cache["num_buckets"])

OUT_DIR = REPO_ROOT / "recacc" / "log" / "notebook_runs"
STEP2_DIR = OUT_DIR / "step2_training"
STEP3_DIR = OUT_DIR / "step3_valuation"
STEP4_DIR = OUT_DIR / "step4_attribution_benchmark"
for p in [OUT_DIR, STEP2_DIR, STEP3_DIR, STEP4_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = torch.load(STEP2_DIR / "all_checkpoints.pt")

train_loader, train_eval_loader, val_loader = make_loaders(
    prepared, batch_size=256, val_ratio=0.2, seed=42
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VAL_SAMPLES = 256
LR = 1e-3

result_rows = []

C:\Users\vanvan\AppData\Local\Temp\ipykernel_18444\2934782795.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cache = torch.load(REPO_ROOT / "recacc" / "log" / "noteboo

In [3]:
for model_name in ["MLP", "VAE", "NCF"]:
    model = build_model(
        name=model_name,
        sparse_fields=prepared.sparse_fields,
        dense_dim=prepared.dense.shape[1],
        num_buckets=NUM_BUCKETS,
    )
    state_path = STEP2_DIR / f"{model_name.lower()}_final_state.pt"
    model.load_state_dict(torch.load(state_path, map_location="cpu"))
    model.to(DEVICE)

    sif_scores = compute_sif(
        model=model,
        val_loader=val_loader,
        train_loader_eval=train_eval_loader,
        max_samples=VAL_SAMPLES,
        device=DEVICE,
    )

    def _builder():
        return build_model(
            name=model_name,
            sparse_fields=prepared.sparse_fields,
            dense_dim=prepared.dense.shape[1],
            num_buckets=NUM_BUCKETS,
        )

    dvf_total, dvf_stage1, dvf_stage2 = compute_dvf_stage(
        model_builder=_builder,
        checkpoints=CHECKPOINTS[model_name],
        train_loader_eval=train_eval_loader,
        val_loader=val_loader,
        max_samples=VAL_SAMPLES,
        lr=LR,
        device=DEVICE,
    )

    rows = []
    ids = sorted(set(list(sif_scores.keys()) + list(dvf_total.keys())))
    for sid in ids:
        rows.append({
            "sample_id": sid,
            "sif": sif_scores.get(sid, 0.0),
            "dvf_total": dvf_total.get(sid, 0.0),
            "dvf_stage1": dvf_stage1.get(sid, 0.0),
            "dvf_stage2": dvf_stage2.get(sid, 0.0),
        })

    save_table(rows, str(STEP3_DIR / f"{model_name.lower()}_valuation.csv"))
    top_pos = sorted(rows, key=lambda x: x["sif"], reverse=True)[:20]
    top_neg = sorted(rows, key=lambda x: x["sif"])[:20]
    save_table(top_pos, str(STEP3_DIR / f"{model_name.lower()}_top_positive_sif.csv"))
    save_table(top_neg, str(STEP3_DIR / f"{model_name.lower()}_top_negative_sif.csv"))

    result_rows.append({
        "model": model_name,
        "num_scored_samples": len(rows),
        "mean_sif": float(sum([r["sif"] for r in rows]) / max(len(rows), 1)),
        "mean_dvf": float(sum([r["dvf_total"] for r in rows]) / max(len(rows), 1)),
    })
    print(model_name, "scored:", len(rows))

save_table(result_rows, str(STEP3_DIR / "valuation_summary.csv"))
result_rows

C:\Users\vanvan\AppData\Local\Temp\ipykernel_18444\259592522.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(state_path, map_location="c

saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\mlp_valuation.csv
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\mlp_top_positive_sif.csv
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\mlp_top_negative_sif.csv
MLP scored: 256
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\vae_valuation.csv
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\vae_top_positive_sif.csv
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\vae_top_negative_sif.csv
VAE scored: 256
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\ncf_valuation.csv
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\ncf_top_positive_sif.csv
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\ncf_top_negative_sif.csv
NCF scored: 256
saved: H:\TZB\recacc\log\notebook_runs\step3_valuation\valuation_summary.csv


[{'model': 'MLP',
  'num_scored_samples': 256,
  'mean_sif': 1.4996123989803891,
  'mean_dvf': 0.0},
 {'model': 'VAE',
  'num_scored_samples': 256,
  'mean_sif': 4.887636279658182,
  'mean_dvf': 4.896272649319144},
 {'model': 'NCF',
  'num_scored_samples': 256,
  'mean_sif': 1.3026807245332748,
  'mean_dvf': 0.0}]

## 小样本归因对比（SHAP vs 影响函数 vs 论文方法）

目标：在很小的数据规模下，比较三类方法的归因准确度与时间开销。

- 归因准确度：用 LOO（leave-one-out）边际贡献作为近似真值，计算与各方法分数的 Spearman 相关。
- 复杂度口径：实测 wall-clock 时间（秒）与理论阶数量级。
- 论文方法：SIF（样本梯度与验证梯度内积）。
- 影响函数：对 $H^{-1}$ 使用对角近似（Fisher diagonal + damping）的经典近似。
- SHAP：采用 Monte Carlo Data Shapley 近似（随机子集边际贡献）。

说明：该实验用于“趋势验证”，不是最终大规模结论。

In [4]:
import hashlib
import json
import random
import time
from functools import lru_cache

# ===== 实验模式总开关 =====
# 可选: "small" | "multiseed"
EXPERIMENT_MODE = "multiseed"
BENCH_MODELS = ["MLP",  "NCF", "VAE"]

# ===== 小样本验证超参（EXPERIMENT_MODE="small"） =====
SMALL_SEED = 55
SMALL_N_TRAIN = 128
SMALL_N_VAL = 512
SMALL_EPOCHS_IN_UTILITY = 20
SMALL_LR_BENCH = 5e-4


SMALL_MC_ITERS_PER_SAMPLE = 100

# ===== 多种子重复超参（EXPERIMENT_MODE="multiseed"） =====
MS_SEEDS = [7, 42, 91]
MS_N_TRAIN = 128
MS_N_VAL = 512
MS_EPOCHS_IN_UTILITY = 20
MS_LR_BENCH = 5e-4


MS_MC_ITERS_PER_SAMPLE = 100

# ===== SHAP 提速配置 =====
# permutation: 更快的近似，通常比逐样本随机子集法更省时
# random_subset: 兼容原始实现
SHAP_ESTIMATOR = "permutation"

# ===== 解释器选择（可作为超参数序列） =====
# 可选: "sif", "influence", "shap"
# 例如只跑 SIF+Influence: ["sif", "influence"]
ATTR_METHODS = ["sif", "influence"]

# ===== 评测参考基准 =====
# loo_spearman: 经典 LOO 相关性（最慢）
# deletion_auc: 快速替代基准（按解释排序删除样本，计算 utility 下降面积）
REFERENCE_BENCHMARK = "deletion_auc"
DEL_FRACS = [0.05, 0.1, 0.2, 0.3]
DEL_RANDOM_REPEATS = 3
_valid_methods = {"sif", "influence", "shap"}
ATTR_METHODS = [m.lower() for m in ATTR_METHODS]
if len(ATTR_METHODS) == 0:
    raise ValueError("ATTR_METHODS 不能为空")
if any(m not in _valid_methods for m in ATTR_METHODS):
    raise ValueError(f"ATTR_METHODS 非法，允许值: {_valid_methods}, 当前: {ATTR_METHODS}")
if REFERENCE_BENCHMARK not in {"loo_spearman", "deletion_auc"}:
    raise ValueError(f"REFERENCE_BENCHMARK 非法: {REFERENCE_BENCHMARK}")

# SHAP 内部 utility 重训轮数；None 表示与 LOO 使用相同 epochs（相关性更稳）
SHAP_UTILITY_EPOCHS = None
# utility 缓存大小（减少重复子集训练）
SHAP_CACHE_MAX = 4096
# utility 方差控制：重复取均值 + 固定子集随机性
UTILITY_AVG_REPEATS = 2
UTILITY_SHUFFLE = False
UTILITY_USE_SUBSET_SEED = True

# ===== LOO 缓存配置（同参数复用，避免重复重训） =====
LOO_CACHE_ENABLE = True
LOO_CACHE_DIR = STEP4_DIR / "loo_cache"
LOO_CACHE_DIR.mkdir(parents=True, exist_ok=True)


def _collect_from_loader(loader, n_samples: int):
    dense_list = []
    y_list = []
    sid_list = []
    sparse_list = {k: [] for k in prepared.sparse_fields}
    c = 0
    for batch in loader:
        dense, sparse, y, sid = _unpack_batch(batch)
        bs = int(dense.shape[0])
        take = min(bs, n_samples - c)
        if take <= 0:
            break
        dense_list.append(dense[:take].detach().cpu())
        y_list.append(y[:take].detach().cpu())
        sid_list.append(sid[:take].detach().cpu())
        for k in prepared.sparse_fields:
            sparse_list[k].append(sparse[k][:take].detach().cpu())
        c += take
        if c >= n_samples:
            break

    dense = torch.cat(dense_list, dim=0)
    labels = torch.cat(y_list, dim=0)
    sample_ids = torch.cat(sid_list, dim=0)
    sparse = {k: torch.cat(v, dim=0) for k, v in sparse_list.items()}
    return PreparedData(
        dense=dense,
        sparse=sparse,
        labels=labels,
        sample_ids=sample_ids,
        sparse_fields=prepared.sparse_fields,
        dense_fields=prepared.dense_fields,
    )


def _loader_from_prepared(p: PreparedData, batch_size=32, shuffle=False):
    ds = RecDataset(p)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def _subset_prepared(p: PreparedData, idxs):
    idxs = torch.tensor(list(idxs), dtype=torch.long)
    return PreparedData(
        dense=p.dense[idxs],
        sparse={k: v[idxs] for k, v in p.sparse.items()},
        labels=p.labels[idxs],
        sample_ids=p.sample_ids[idxs],
        sparse_fields=p.sparse_fields,
        dense_fields=p.dense_fields,
    )


def _spearman(a, b):
    sa = pd.Series(a).rank(method="average").to_numpy()
    sb = pd.Series(b).rank(method="average").to_numpy()
    if np.std(sa) == 0 or np.std(sb) == 0:
        return 0.0
    return float(np.corrcoef(sa, sb)[0, 1])


def _stable_ids_hash(ids):
    raw = ",".join([str(int(x)) for x in ids]).encode("utf-8")
    return hashlib.sha1(raw).hexdigest()[:16]


def _build_loo_cache_path(bench_model, seed, n_train, n_val, epochs_in_utility, lr_bench, ids):
    payload = {
        "bench_model": str(bench_model),
        "seed": int(seed),
        "n_train": int(n_train),
        "n_val": int(n_val),
        "epochs_in_utility": int(epochs_in_utility),
        "lr_bench": float(lr_bench),
        "utility_avg_repeats": int(UTILITY_AVG_REPEATS),
        "utility_shuffle": bool(UTILITY_SHUFFLE),
        "utility_use_subset_seed": bool(UTILITY_USE_SUBSET_SEED),
        "ids_hash": _stable_ids_hash(ids),
    }
    key = hashlib.sha1(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:24]
    return LOO_CACHE_DIR / f"loo_{str(bench_model).lower()}_{key}.pt"


def _run_one_setting(seed, n_train, n_val, epochs_in_utility, mc_iters_per_sample, lr_bench):
    seed_everything(seed)
    random.seed(seed)
    np.random.seed(seed)

    train_small = _collect_from_loader(train_eval_loader, n_train)
    val_small = _collect_from_loader(val_loader, n_val)
    train_small_loader = _loader_from_prepared(train_small, batch_size=16, shuffle=True)
    val_small_loader = _loader_from_prepared(val_small, batch_size=32, shuffle=False)

    ids = [int(x.item()) for x in train_small.sample_ids]
    id_to_pos = {sid: i for i, sid in enumerate(ids)}
    full_idxs = list(range(len(ids)))

    all_rows = []

    for bench_model in BENCH_MODELS:
        print("\n===", bench_model, "seed", seed, "===")

        model_base = build_model(
            name=bench_model,
            sparse_fields=prepared.sparse_fields,
            dense_dim=prepared.dense.shape[1],
            num_buckets=NUM_BUCKETS,
        )
        state_path = STEP2_DIR / f"{bench_model.lower()}_final_state.pt"
        model_base.load_state_dict(torch.load(state_path, map_location="cpu"))
        model_base.to(DEVICE)

        base_state = {k: v.detach().cpu().clone() for k, v in model_base.state_dict().items()}

        def _stable_subset_seed(selected_idxs, rep):
            key = sorted([int(x) for x in selected_idxs])
            acc = 0
            for t, x in enumerate(key):
                acc = (acc + (t + 1) * (x + 17)) % 2147483647
            name_acc = sum([ord(ch) for ch in bench_model])
            return int((seed * 1000003 + acc * 1315423911 + rep * 2654435761 + name_acc) % 2147483647)

        def utility_auc(selected_idxs, epochs_override=None):
            selected_idxs = list(selected_idxs)
            if len(selected_idxs) == 0:
                return 0.5
            use_epochs = epochs_in_utility if epochs_override is None else int(epochs_override)
            sub = _subset_prepared(train_small, selected_idxs)
            vals = []
            for rep in range(max(int(UTILITY_AVG_REPEATS), 1)):
                if UTILITY_USE_SUBSET_SEED:
                    s2 = _stable_subset_seed(selected_idxs, rep)
                    seed_everything(s2)
                    random.seed(s2)
                    np.random.seed(s2)

                sub_loader = _loader_from_prepared(sub, batch_size=16, shuffle=UTILITY_SHUFFLE)

                m = build_model(
                    name=bench_model,
                    sparse_fields=prepared.sparse_fields,
                    dense_dim=prepared.dense.shape[1],
                    num_buckets=NUM_BUCKETS,
                )
                m.load_state_dict(base_state)
                _hist, _snap = train_model(
                    m,
                    sub_loader,
                    val_small_loader,
                    epochs=use_epochs,
                    lr=lr_bench,
                    device=DEVICE,
                    verbose=False,
                )
                metrics = evaluate_model(m, val_small_loader, device=DEVICE)
                vals.append(float(metrics["auc"]))
            return float(np.mean(vals))

        @lru_cache(maxsize=SHAP_CACHE_MAX)
        def utility_auc_cached(sorted_idxs_key):
            e = epochs_in_utility if SHAP_UTILITY_EPOCHS is None else SHAP_UTILITY_EPOCHS
            return utility_auc(list(sorted_idxs_key), epochs_override=e)

        full_u = utility_auc(full_idxs)
        loo_cache_hit = False
        loo_time = 0.0
        gt = {}

        if REFERENCE_BENCHMARK == "loo_spearman":
            loo_cache_path = _build_loo_cache_path(
                bench_model=bench_model,
                seed=seed,
                n_train=n_train,
                n_val=n_val,
                epochs_in_utility=epochs_in_utility,
                lr_bench=lr_bench,
                ids=ids,
            )

            if LOO_CACHE_ENABLE and loo_cache_path.exists():
                cache_blob = torch.load(loo_cache_path, map_location="cpu")
                cached_ids = [int(x) for x in cache_blob.get("ids", [])]
                if cached_ids == ids:
                    gt = {int(k): float(v) for k, v in cache_blob.get("gt", {}).items()}
                    loo_time = float(cache_blob.get("loo_time_sec", 0.0))
                    loo_cache_hit = True
                    print("LOO cache hit:", loo_cache_path.name)

            if not loo_cache_hit:
                st = time.perf_counter()
                for i, sid in enumerate(ids):
                    keep = [j for j in full_idxs if j != i]
                    gt[sid] = full_u - utility_auc(keep)
                loo_time = time.perf_counter() - st

                if LOO_CACHE_ENABLE:
                    torch.save(
                        {
                            "gt": {int(k): float(v) for k, v in gt.items()},
                            "ids": [int(x) for x in ids],
                            "loo_time_sec": float(loo_time),
                        },
                        loo_cache_path,
                    )
                    print("LOO cache saved:", loo_cache_path.name)

        def _deletion_auc_from_scores(scores):
            ranked = sorted(ids, key=lambda sid: float(scores.get(sid, 0.0)), reverse=True)
            deltas = []
            for frac in DEL_FRACS:
                k = max(1, int(round(len(ids) * float(frac))))
                removed = set(ranked[:k])
                keep = [j for j in full_idxs if ids[j] not in removed]
                u = utility_auc(keep)
                deltas.append(float(full_u - u))

            x = np.array(DEL_FRACS, dtype=float)
            y = np.array(deltas, dtype=float)
            area = float(np.trapz(y, x))

            rnd_vals = []
            for _ in range(max(int(DEL_RANDOM_REPEATS), 1)):
                rr = ids.copy()
                random.shuffle(rr)
                rd = []
                for frac in DEL_FRACS:
                    k = max(1, int(round(len(ids) * float(frac))))
                    removed = set(rr[:k])
                    keep = [j for j in full_idxs if ids[j] not in removed]
                    u = utility_auc(keep)
                    rd.append(float(full_u - u))
                rnd_vals.append(float(np.trapz(np.array(rd, dtype=float), x)))

            random_area = float(np.mean(rnd_vals))
            return area - random_area

        sif_scores = {}
        if_scores = {}
        shap_scores = {}
        sif_time = 0.0
        if_time = 0.0
        shap_time = 0.0
        shap_note = ""

        if "sif" in ATTR_METHODS:
            st = time.perf_counter()
            sif_scores = compute_sif(
                model=model_base,
                val_loader=val_small_loader,
                train_loader_eval=train_small_loader,
                max_samples=len(ids),
                device=DEVICE,
            )
            sif_time = time.perf_counter() - st

        if "influence" in ATTR_METHODS:
            st = time.perf_counter()
            vg = val_grad(model_base, val_small_loader, device=DEVICE)
            all_grads = {}
            for batch in train_small_loader:
                dense, sparse, y, sid = _unpack_batch(batch)
                bs = dense.shape[0]
                for i in range(bs):
                    sid_i = int(sid[i].item())
                    gi = sample_grad(
                        model_base,
                        dense[i : i + 1],
                        {k: v[i : i + 1] for k, v in sparse.items()},
                        y[i : i + 1],
                        device=DEVICE,
                    )
                    all_grads[sid_i] = gi

            damp = 1e-3
            diag = None
            for sid in ids:
                g2 = all_grads[sid] * all_grads[sid]
                diag = g2 if diag is None else (diag + g2)
            diag = diag / max(len(ids), 1) + damp

            for sid in ids:
                gi = all_grads[sid]
                if_scores[sid] = float(-(vg * gi / diag).sum().item())
            if_time = time.perf_counter() - st

        if "shap" in ATTR_METHODS:
            st = time.perf_counter()
            shap_scores = {sid: 0.0 for sid in ids}

            if SHAP_ESTIMATOR == "permutation":
                for _ in range(mc_iters_per_sample):
                    perm = full_idxs.copy()
                    random.shuffle(perm)
                    prefix = []
                    prev_u = 0.5
                    for idx in perm:
                        prefix.append(idx)
                        key = tuple(sorted(prefix))
                        cur_u = utility_auc_cached(key)
                        sid = ids[idx]
                        shap_scores[sid] += (cur_u - prev_u)
                        prev_u = cur_u
                for sid in ids:
                    shap_scores[sid] /= max(mc_iters_per_sample, 1)
                shap_note = f"O(M·N·T_train), permutation, M={mc_iters_per_sample}, e={(epochs_in_utility if SHAP_UTILITY_EPOCHS is None else SHAP_UTILITY_EPOCHS)}, r={UTILITY_AVG_REPEATS}"
            elif SHAP_ESTIMATOR == "random_subset":
                for sid in ids:
                    i = id_to_pos[sid]
                    acc = 0.0
                    for _ in range(mc_iters_per_sample):
                        pool = [j for j in full_idxs if j != i]
                        k = random.randint(0, len(pool))
                        S = random.sample(pool, k) if k > 0 else []
                        key0 = tuple(sorted(S))
                        key1 = tuple(sorted(S + [i]))
                        u0 = utility_auc_cached(key0)
                        u1 = utility_auc_cached(key1)
                        acc += (u1 - u0)
                    shap_scores[sid] = acc / max(mc_iters_per_sample, 1)
                shap_note = f"O(M·N·T_train), random_subset, M={mc_iters_per_sample}, e={(epochs_in_utility if SHAP_UTILITY_EPOCHS is None else SHAP_UTILITY_EPOCHS)}, r={UTILITY_AVG_REPEATS}"
            else:
                raise ValueError(f"Unsupported SHAP_ESTIMATOR: {SHAP_ESTIMATOR}")

            shap_time = time.perf_counter() - st

        rows = []
        method_records = []
        if "sif" in ATTR_METHODS:
            method_records.append(("Paper-SIF", sif_scores, sif_time, "O(N·P + V·P)"))
        if "influence" in ATTR_METHODS:
            method_records.append(("Influence(diag)", if_scores, if_time, "O(N·P + V·P)"))
        if "shap" in ATTR_METHODS:
            method_records.append(("SHAP-MC(Data)", shap_scores, shap_time, shap_note))

        for name, scores, spent, complexity in method_records:
            svec = [float(scores.get(sid, 0.0)) for sid in ids]
            if REFERENCE_BENCHMARK == "loo_spearman":
                gt_vec = [gt[sid] for sid in ids]
                metric_value = _spearman(gt_vec, svec)
                metric_name = "spearman_vs_loo"
            else:
                metric_value = _deletion_auc_from_scores(scores)
                metric_name = "deletion_auc_minus_random"
            rows.append(
                {
                    "seed": seed,
                    "model": bench_model,
                    "method": name,
                    "spearman_vs_loo": float(metric_value),
                    "metric_name": metric_name,
                    "time_sec": float(spent),
                    "loo_time_sec": float(loo_time),
                    "loo_cache_hit": bool(loo_cache_hit),
                    "num_train_samples": len(ids),
                    "num_val_samples": int(val_small.labels.shape[0]),
                    "note": complexity,
                    "state_path": str(state_path),
                }
            )

        model_df = pd.DataFrame(rows).sort_values("spearman_vs_loo", ascending=False)
        model_path = STEP4_DIR / f"attribution_benchmark_{bench_model.lower()}_seed{seed}.csv"
        save_table(model_df, str(model_path))
        print("saved:", model_path)
        all_rows.extend(rows)

    return pd.DataFrame(all_rows)


def _build_time_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=["step", "time_sec", "ratio"])

    # 每行的 loo_time_sec 相同地重复在三个 method 行中，先按 seed+model 去重再求和
    loo_total = float(df[["seed", "model", "loo_time_sec"]].drop_duplicates()["loo_time_sec"].sum())
    sif_total = float(df.loc[df["method"] == "Paper-SIF", "time_sec"].sum())
    if_total = float(df.loc[df["method"] == "Influence(diag)", "time_sec"].sum())
    shap_total = float(df.loc[df["method"] == "SHAP-MC(Data)", "time_sec"].sum())

    out = pd.DataFrame([
        {"step": "LOO-reference", "time_sec": loo_total},
        {"step": "Paper-SIF", "time_sec": sif_total},
        {"step": "Influence(diag)", "time_sec": if_total},
        {"step": "SHAP-MC(Data)", "time_sec": shap_total},
    ])
    total = float(out["time_sec"].sum())
    out["ratio"] = out["time_sec"] / total if total > 0 else 0.0
    out = out.sort_values("time_sec", ascending=False).reset_index(drop=True)
    return out


if EXPERIMENT_MODE == "small":
    all_df = _run_one_setting(
        seed=SMALL_SEED,
        n_train=SMALL_N_TRAIN,
        n_val=SMALL_N_VAL,
        epochs_in_utility=SMALL_EPOCHS_IN_UTILITY,
        mc_iters_per_sample=SMALL_MC_ITERS_PER_SAMPLE,
        lr_bench=SMALL_LR_BENCH,
    ).sort_values(["model", "spearman_vs_loo"], ascending=[True, False])

    all_path = STEP4_DIR / "attribution_benchmark_all_models.csv"
    save_table(all_df, str(all_path))
    time_breakdown_df = _build_time_breakdown(all_df)
    time_breakdown_path = STEP4_DIR / "time_breakdown_small.csv"
    save_table(time_breakdown_df, str(time_breakdown_path))
    print("saved:", all_path)
    print("saved:", time_breakdown_path)
    print("\n[Time breakdown]")
    print(time_breakdown_df)
    all_df

elif EXPERIMENT_MODE == "multiseed":
    frames = []
    for seed in MS_SEEDS:
        df_seed = _run_one_setting(
            seed=seed,
            n_train=MS_N_TRAIN,
            n_val=MS_N_VAL,
            epochs_in_utility=MS_EPOCHS_IN_UTILITY,
            mc_iters_per_sample=MS_MC_ITERS_PER_SAMPLE,
            lr_bench=MS_LR_BENCH,
        )
        frames.append(df_seed)

    detail_df = pd.concat(frames, ignore_index=True)
    detail_path = STEP4_DIR / "attribution_benchmark_multiseed_detail.csv"
    save_table(detail_df, str(detail_path))
    time_breakdown_df = _build_time_breakdown(detail_df)
    time_breakdown_path = STEP4_DIR / "time_breakdown_multiseed.csv"
    save_table(time_breakdown_df, str(time_breakdown_path))

    agg_df = (
        detail_df.groupby(["model", "method"], as_index=False)
        .agg(
            spearman_mean=("spearman_vs_loo", "mean"),
            spearman_std=("spearman_vs_loo", "std"),
            time_mean_sec=("time_sec", "mean"),
            time_std_sec=("time_sec", "std"),
            loo_time_mean_sec=("loo_time_sec", "mean"),
            loo_time_std_sec=("loo_time_sec", "std"),
            runs=("seed", "count"),
        )
        .sort_values(["model", "spearman_mean"], ascending=[True, False])
    )
    agg_path = STEP4_DIR / "attribution_benchmark_multiseed_summary.csv"
    save_table(agg_df, str(agg_path))

    print("saved:", detail_path)
    print("saved:", agg_path)
    print("saved:", time_breakdown_path)
    print("\n[Time breakdown]")
    print(time_breakdown_df)
    agg_df

else:
    raise ValueError(f"Unsupported EXPERIMENT_MODE: {EXPERIMENT_MODE}")


=== MLP seed 7 ===


C:\Users\vanvan\AppData\Local\Temp\ipykernel_18444\1726704023.py:180: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_base.load_state_dict(torch.load(state_path, map_loc

saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_mlp_seed7.csv
saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_mlp_seed7.csv

=== NCF seed 7 ===


C:\Users\vanvan\AppData\Local\Temp\ipykernel_18444\1726704023.py:180: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_base.load_state_dict(torch.load(state_path, map_loc

KeyboardInterrupt: 

## 多种子综合结果表

将多种子明细汇总为三类结果：

- 方法总体均值/方差（跨模型）
- 模型-方法宽表（便于论文主表）
- 综合排名表（准确度与时间联合排序）

In [ ]:
from pathlib import Path

# 若当前会话未保留 detail_df，则从文件读取
if "detail_df" not in globals() or detail_df is None or len(detail_df) == 0:
    detail_path = STEP4_DIR / "attribution_benchmark_multiseed_detail.csv"
    if not detail_path.exists():
        raise FileNotFoundError(
            f"未找到多种子明细文件: {detail_path}。请先将 EXPERIMENT_MODE 设为 'multiseed' 并运行第 5 个代码单元。"
        )
    detail_df = pd.read_csv(detail_path)

# 1) 跨模型总体表（按方法聚合）
overall_df = (
    detail_df.groupby("method", as_index=False)
    .agg(
        spearman_mean=("spearman_vs_loo", "mean"),
        spearman_std=("spearman_vs_loo", "std"),
        time_mean_sec=("time_sec", "mean"),
        time_std_sec=("time_sec", "std"),
        runs=("seed", "count"),
    )
)

# 2) 模型-方法宽表（论文展示友好）
wide_df = (
    detail_df.groupby(["model", "method"], as_index=False)
    .agg(
        spearman_mean=("spearman_vs_loo", "mean"),
        spearman_std=("spearman_vs_loo", "std"),
        time_mean_sec=("time_sec", "mean"),
        time_std_sec=("time_sec", "std"),
    )
)
wide_df = wide_df.pivot(index="model", columns="method", values=["spearman_mean", "spearman_std", "time_mean_sec", "time_std_sec"])
wide_df.columns = [f"{m}__{k}" for k, m in wide_df.columns]
wide_df = wide_df.reset_index()

# 3) 综合排名表（准确度高 + 时间低）
rank_df = (
    detail_df.groupby("method", as_index=False)
    .agg(
        spearman_mean=("spearman_vs_loo", "mean"),
        time_mean_sec=("time_sec", "mean"),
    )
)
rank_df["rank_spearman"] = rank_df["spearman_mean"].rank(ascending=False, method="min")
rank_df["rank_time"] = rank_df["time_mean_sec"].rank(ascending=True, method="min")
rank_df["rank_total"] = rank_df["rank_spearman"] + rank_df["rank_time"]
rank_df = rank_df.sort_values(["rank_total", "rank_spearman"]).reset_index(drop=True)

# 保存综合表
overall_path = STEP4_DIR / "attribution_benchmark_multiseed_overall.csv"
wide_path = STEP4_DIR / "attribution_benchmark_multiseed_wide.csv"
rank_path = STEP4_DIR / "attribution_benchmark_multiseed_rank.csv"

save_table(overall_df, str(overall_path))
save_table(wide_df, str(wide_path))
save_table(rank_df, str(rank_path))

print("saved:", overall_path)
print("saved:", wide_path)
print("saved:", rank_path)

print("\n[Overall]")
overall_df


saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_multiseed_overall.csv
saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_multiseed_wide.csv
saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_multiseed_rank.csv
saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_multiseed_overall.csv
saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_multiseed_wide.csv
saved: H:\TZB\recacc\log\notebook_runs\step4_attribution_benchmark\attribution_benchmark_multiseed_rank.csv

[Overall]


,method,spearman_mean,spearman_std,time_mean_sec,time_std_sec,runs
0,Influence(diag),-0.016875,0.003215,1.262185,0.629047,9
1,Paper-SIF,0.008122,0.003728,0.773152,0.280042,9


## 综合验证体系（问题驱动）

目标：把“方法是否可信”拆成一组可回答的问题，形成统一验证协议。

### A. 方法前置问题（每个方法都要回答）

1. 有效性问题
- 该方法是否稳定优于随机删除基线？（`deletion_auc_minus_random > 0`）
- 在不同模型上是否都保持同方向结论？

2. 一致性问题
- 与参考基准（`loo_spearman` 或 `deletion_auc`）是否同方向？
- 若出现反向（如稳定负值），是符号约定问题还是方法失效？

3. 稳定性问题
- 多 seed 下方差是否可控？
- 模型间波动是否过大（泛化风险）？

4. 效率问题
- 单次评测时间是否满足预算？
- 相比其它方法是否有明显性价比优势？

5. 可解释性问题
- 输出分数是否可排序且可复现？
- 改动少量超参后结论是否保持？

### B. 针对本项目三类方法的专属问题

- Paper-SIF
  问题：是否具备“快速且稳定弱正收益”的性质？

- Influence(diag)
  问题：是否存在方向约定不一致（稳定负值）？取反后是否恢复有效性？

- SHAP-MC(Data)
  问题：在固定预算下，是否明显优于随机且不被方差主导？

### C. 判定规则（建议）

- `strong`: 均值 > 0.05 且 std < 0.03
- `moderate`: 均值 > 0.02 且 std < 0.05
- `weak`: 均值 > 0 且非显著波动
- `invalid`: 均值 <= 0 或方向长期反向

In [5]:
# 综合验证打分：基于当前 detail_df/all_df 自动生成方法结论表
from pathlib import Path
import numpy as np
import pandas as pd

# 读取最新明细（优先使用内存变量）
if "detail_df" in globals() and detail_df is not None and len(detail_df) > 0:
    eval_df = detail_df.copy()
elif "all_df" in globals() and all_df is not None and len(all_df) > 0:
    eval_df = all_df.copy()
else:
    p_multi = STEP4_DIR / "attribution_benchmark_multiseed_detail.csv"
    p_small = STEP4_DIR / "attribution_benchmark_all_models.csv"
    if p_multi.exists():
        eval_df = pd.read_csv(p_multi)
    elif p_small.exists():
        eval_df = pd.read_csv(p_small)
    else:
        raise FileNotFoundError("未找到评测结果，请先运行第 5 个代码单元。")

if "metric_name" not in eval_df.columns:
    eval_df["metric_name"] = "spearman_vs_loo"

metric_name = str(eval_df["metric_name"].dropna().iloc[0]) if len(eval_df) > 0 else "unknown"

agg = (
    eval_df.groupby("method", as_index=False)
    .agg(
        score_mean=("spearman_vs_loo", "mean"),
        score_std=("spearman_vs_loo", "std"),
        time_mean_sec=("time_sec", "mean"),
        time_std_sec=("time_sec", "std"),
        models=("model", "nunique"),
        runs=("seed", "count"),
    )
)

agg["score_std"] = agg["score_std"].fillna(0.0)
agg["time_std_sec"] = agg["time_std_sec"].fillna(0.0)

# 通用分档
def _tier(mean_v, std_v):
    if mean_v > 0.05 and std_v < 0.03:
        return "strong"
    if mean_v > 0.02 and std_v < 0.05:
        return "moderate"
    if mean_v > 0:
        return "weak"
    return "invalid"

agg["tier"] = agg.apply(lambda r: _tier(float(r["score_mean"]), float(r["score_std"])), axis=1)

# 问题化判定（每个方法对应核心问题）
questions = []
for _, r in agg.iterrows():
    m = str(r["method"])
    mean_v = float(r["score_mean"])
    std_v = float(r["score_std"])
    tier = str(r["tier"])

    if m == "Paper-SIF":
        q = "是否具备快速且稳定弱正收益？"
        a = "是" if mean_v > 0 else "否"
    elif m == "Influence(diag)":
        q = "是否存在方向约定不一致（稳定负值）？"
        a = "是" if mean_v < 0 else "否"
    elif m == "SHAP-MC(Data)":
        q = "固定预算下是否明显优于随机？"
        a = "是" if mean_v > 0 else "否"
    else:
        q = "该方法是否稳定优于随机基线？"
        a = "是" if mean_v > 0 else "否"

    questions.append({
        "method": m,
        "question": q,
        "answer": a,
        "score_mean": mean_v,
        "score_std": std_v,
        "tier": tier,
        "time_mean_sec": float(r["time_mean_sec"]),
        "metric_name": metric_name,
    })

qa_df = pd.DataFrame(questions).sort_values(["tier", "score_mean"], ascending=[True, False])

qa_path = STEP4_DIR / "validation_protocol_summary.csv"
save_table(qa_df, str(qa_path))

print("metric_name:", metric_name)
print("saved:", qa_path)
qa_df

FileNotFoundError: 未找到评测结果，请先运行第 5 个代码单元。